# Project FORESIGHT — Notebook 02: Seasonal-Naive Baseline

Builds a weekly, SKU-level demand baseline using a seasonal-naive method, evaluated with
proper rolling-origin (walk-forward) backtesting. This baseline is the bar the model in
notebook 03 must beat.

**Forecast horizon: 6 weeks.** Chosen because it gives the operations team enough lead time to
act on reorder/markdown decisions while staying short enough that seasonal-naive lookups
remain meaningful.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
HORIZON = 6          # weeks
SEASON_LENGTH = 52   # weeks, for seasonal-naive lookback

df = pd.read_csv(PROCESSED_DIR / "modeling_data.csv", parse_dates=["Date"])
df.shape

(36550, 29)

## 1. Build the Weekly SKU-Level Demand Panel

Daily sales are aggregated to ISO-style Monday-start weeks. Calendar attributes for each week
are derived as: any holiday in the week, any promotion event in the week, and the week's month/
quarter/season (all known in advance, since the calendar is fixed).

In [2]:
df["week_start"] = df["Date"] - pd.to_timedelta(df["Date"].dt.dayofweek, unit="D")

weekly = df.groupby(["SKU", "week_start"]).agg(
    Units_Sold=("Units_Sold", "sum"),
    Revenue=("Revenue", "sum"),
    is_holiday_week=("is_holiday", "max"),
    month=("month", "first"),
    quarter=("quarter", "first"),
    season=("season", "first"),
    Category=("Category", "first"),
    Subcategory=("Subcategory", "first"),
    Launch_Date=("Launch_Date", "first"),
    Gross_Margin_Per_Unit=("Gross_Margin_Per_Unit", "first"),
).reset_index()

# A week has a promotion event if any day in it carries a named promotion_event
promo_flag = df.groupby(["SKU", "week_start"])["promotion_event"].apply(
    lambda s: int((s != "None").any())).rename("has_promo_event")
weekly = weekly.merge(promo_flag, on=["SKU", "week_start"])

weekly = weekly.sort_values(["SKU", "week_start"]).reset_index(drop=True)

# Global week index (same calendar for every SKU, so one ordinal series works for all)
week_order = sorted(weekly["week_start"].unique())
week_index_map = {w: i for i, w in enumerate(week_order)}
weekly["week_index"] = weekly["week_start"].map(week_index_map)

print(f"Weekly panel: {weekly.shape[0]} rows, {weekly['SKU'].nunique()} SKUs, "
      f"{weekly['week_index'].max()+1} distinct weeks "
      f"({weekly['week_start'].min().date()} to {weekly['week_start'].max().date()})")
weekly.head(3)

Weekly panel: 5250 rows, 50 SKUs, 105 distinct weeks (2024-01-01 to 2025-12-29)


,SKU,week_start,Units_Sold,Revenue,is_holiday_week,month,quarter,season,Category,Subcategory,Launch_Date,Gross_Margin_Per_Unit,has_promo_event,week_index
0,SKU001,2024-01-01,106,388389.30,0,1,Q1,Winter,Furniture,Chair,2022-04-09,1905.6,1,0
1,SKU001,2024-01-08,85,311444.25,0,1,Q1,Winter,Furniture,Chair,2022-04-09,1905.6,1,1
2,SKU001,2024-01-15,117,428693.85,0,1,Q1,Winter,Furniture,Chair,2022-04-09,1905.6,1,2


## 2. Seasonal-Naive Baseline

For each SKU and week, the seasonal-naive prediction is the actual demand observed exactly
**52 weeks earlier** for that SKU. This only uses past information relative to any given week,
so it is inherently walk-forward safe.

For the first 52 weeks of history (where no year-ago value exists yet), the naive baseline
falls back to the previous week's actual value — this limitation is documented here rather
than hidden.

In [3]:
weekly = weekly.sort_values(["SKU", "week_index"]).reset_index(drop=True)
weekly["seasonal_naive"] = weekly.groupby("SKU")["Units_Sold"].shift(SEASON_LENGTH)
weekly["naive_fallback"] = weekly.groupby("SKU")["Units_Sold"].shift(1)
n_fallback = weekly["seasonal_naive"].isna().sum()
weekly["baseline_pred"] = weekly["seasonal_naive"].fillna(weekly["naive_fallback"])

print(f"Rows using the year-ago seasonal value: {weekly['seasonal_naive'].notna().sum()}")
print(f"Rows using the previous-week fallback (first {SEASON_LENGTH} weeks of each SKU's history): {n_fallback}")

Rows using the year-ago seasonal value: 2650
Rows using the previous-week fallback (first 52 weeks of each SKU's history): 2600


## 3. Rolling-Origin (Walk-Forward) Backtest

Multiple chronological origins are evaluated, each forecasting the next 6 weeks using only
information available up to that origin. This avoids the single random train/test split that
would otherwise overstate accuracy on time-series data.

In [4]:
def wape(actual, pred):
    actual, pred = np.asarray(actual, dtype=float), np.asarray(pred, dtype=float)
    return np.abs(pred - actual).sum() / actual.sum()

def bias(actual, pred):
    actual, pred = np.asarray(actual, dtype=float), np.asarray(pred, dtype=float)
    return (pred - actual).sum() / actual.sum()

max_week = weekly["week_index"].max()
origin_min = SEASON_LENGTH               # ensures the whole horizon has a valid seasonal lookback
origin_max = max_week - HORIZON
origins = list(range(origin_min, origin_max + 1, HORIZON))
print(f"Backtest origins (rolling, non-overlapping {HORIZON}-week windows): {len(origins)}")
print(origins)

Backtest origins (rolling, non-overlapping 6-week windows): 8
[52, 58, 64, 70, 76, 82, 88, 94]


In [5]:
results = []
for origin in origins:
    window = weekly[(weekly["week_index"] > origin) & (weekly["week_index"] <= origin + HORIZON)]
    if window.empty or window["baseline_pred"].isna().any():
        continue
    results.append({
        "origin_week_index": origin,
        "origin_date": week_order[origin].date(),
        "test_start": week_order[origin + 1].date(),
        "test_end": week_order[min(origin + HORIZON, max_week)].date(),
        "WAPE": wape(window["Units_Sold"], window["baseline_pred"]),
        "Bias": bias(window["Units_Sold"], window["baseline_pred"]),
        "n_obs": len(window),
    })

baseline_backtest = pd.DataFrame(results)
baseline_backtest

,origin_week_index,origin_date,test_start,test_end,WAPE,Bias,n_obs
0,52,2024-12-30,2025-01-06,2025-02-10,0.109939,0.002874,300
1,58,2025-02-10,2025-02-17,2025-03-24,0.091368,-0.007024,300
2,64,2025-03-24,2025-03-31,2025-05-05,0.104157,0.004858,300
3,70,2025-05-05,2025-05-12,2025-06-16,0.093435,0.006193,300
4,76,2025-06-16,2025-06-23,2025-07-28,0.111480,0.001573,300
5,82,2025-07-28,2025-08-04,2025-09-08,0.118818,0.004823,300
6,88,2025-09-08,2025-09-15,2025-10-20,0.115163,0.001897,300
7,94,2025-10-20,2025-10-27,2025-12-01,0.111420,-0.001160,300


In [6]:
overall_wape = wape(
    weekly.loc[weekly["week_index"] > origin_min, "Units_Sold"].where(weekly["week_index"] <= origin_max + HORIZON).dropna(),
    weekly.loc[weekly["week_index"] > origin_min, "baseline_pred"].where(weekly["week_index"] <= origin_max + HORIZON).dropna(),
) if False else None

# Simpler, robust overall computation: pool every row used across all backtest windows
mask = (weekly["week_index"] > origin_min) & (weekly["week_index"] <= origin_max + HORIZON)
overall_wape = wape(weekly.loc[mask, "Units_Sold"], weekly.loc[mask, "baseline_pred"])
overall_bias = bias(weekly.loc[mask, "Units_Sold"], weekly.loc[mask, "baseline_pred"])

print(f"Baseline overall WAPE across all backtest windows: {overall_wape:.4f}")
print(f"Baseline overall Bias across all backtest windows: {overall_bias:+.4f}")
print("(Bias > 0 means the baseline tends to over-forecast; < 0 means it tends to under-forecast.)")

Baseline overall WAPE across all backtest windows: 0.1180
Baseline overall Bias across all backtest windows: +0.0136
(Bias > 0 means the baseline tends to over-forecast; < 0 means it tends to under-forecast.)


## 4. Baseline Visuals

In [7]:
fig = px.bar(baseline_backtest, x="test_start", y="WAPE",
             title="Seasonal-Naive Baseline: WAPE per Rolling-Origin Backtest Window",
             labels={"test_start": "Test window start", "WAPE": "WAPE"},
             template="plotly_white")
fig.add_hline(y=overall_wape, line_dash="dash", line_color="red",
              annotation_text=f"Overall WAPE = {overall_wape:.3f}")
fig.show()

In [8]:
# Representative SKUs: highest-revenue, a mid-tier, and a low-volume SKU
sku_totals = weekly.groupby("SKU")["Units_Sold"].sum().sort_values(ascending=False)
rep_skus = [sku_totals.index[0], sku_totals.index[len(sku_totals)//2], sku_totals.index[-1]]

fig = go.Figure()
for sku in rep_skus:
    sub = weekly[weekly["SKU"] == sku]
    fig.add_trace(go.Scatter(x=sub["week_start"], y=sub["Units_Sold"], mode="lines",
                              name=f"{sku} actual", legendgroup=sku))
    fig.add_trace(go.Scatter(x=sub["week_start"], y=sub["baseline_pred"], mode="lines",
                              name=f"{sku} seasonal-naive", line=dict(dash="dot"), legendgroup=sku))

fig.update_layout(title="Weekly Actual Demand vs Seasonal-Naive Baseline (representative SKUs)",
                   xaxis_title="Week", yaxis_title="Units Sold", template="plotly_white", height=480)
fig.show()

## 5. Save Baseline Outputs

The weekly panel (with the baseline prediction already attached) and the backtest results are
saved for notebook 03 to build on and compare against.

In [9]:
weekly.to_csv(PROCESSED_DIR / "weekly_demand_panel.csv", index=False)
baseline_backtest.to_csv(PROCESSED_DIR / "baseline_backtest_results.csv", index=False)

with open(REPORTS_DIR / "baseline_summary.txt", "w") as f:
    f.write("FORESIGHT — SEASONAL-NAIVE BASELINE SUMMARY\n")
    f.write("=" * 45 + "\n\n")
    f.write(f"Forecast horizon: {HORIZON} weeks\n")
    f.write(f"Seasonal lookback: {SEASON_LENGTH} weeks\n")
    f.write(f"Backtest windows: {len(baseline_backtest)}\n")
    f.write(f"Overall WAPE: {overall_wape:.4f}\n")
    f.write(f"Overall Bias: {overall_bias:+.4f}\n")
    f.write(f"Rows relying on previous-week fallback (no year-ago value yet): {n_fallback}\n")

print("Saved:")
print(" - data/processed/weekly_demand_panel.csv")
print(" - data/processed/baseline_backtest_results.csv")
print(" - reports/baseline_summary.txt")

Saved:
 - data/processed/weekly_demand_panel.csv
 - data/processed/baseline_backtest_results.csv
 - reports/baseline_summary.txt
